# LOGAN: six-frame inheritance, discrimination and Δ

**Draft caption.** Six-frame experiment on 8.82 million complete Prodigal genes (1% deterministic sample, all 90 shards). (a) Spearman correlation between the value in Prodigal's selected frame and in each of the five alternative frames (same strand +1/+2, opposite strand a0–a2): length and AA entropy are partly shared between frames; 3Di entropy and MI are not. (b) Fraction of genes for which each feature ranks the selected frame first, in the top two or top three of six (5,722,417 genes with all five alternatives; ties counted fractionally; dotted lines, chance 1/6, 1/3, 1/2). (c–g) Distribution of Δ = selected-frame value − median of the five alternatives (central 99% shown); the fraction of genes with Δ > 0 is computed over all genes in the sample.

**Outputs:** `../images/figL05_six_frame.pdf` (vector) and `../images/figL05_six_frame.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL05_six_frame_inheritance.tsv.gz` (25 rows; sha256 `7bd4bfbe4dc2…`): Spearman rho, selected vs each alternative frame. Source on Setonix (`$LOGAN_ROOT`): `qc/six_frame_final/inheritance.tsv`.
- `../data/figL05_six_frame_discrimination.tsv.gz` (10 rows; sha256 `fdfaf51e8d06…`): tie-aware top-k rates and chance. Source on Setonix (`$LOGAN_ROOT`): `qc/six_frame_final/discrimination.tsv`.
- `../data/figL05_six_frame_delta.tsv.gz` (600 rows; sha256 `76589b3cfe6e…`): histogram of Δ (selected − median of 5 alternatives), central 99%, 120 bins; genes with all 5. Source on Setonix (`$LOGAN_ROOT`): `qc/six_frame_final/delta_sample.parquet`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL05_six_frame.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
FEAT = {"aa_len": "ORF length", "aa_entropy": "AA entropy", "three_di_entropy": "3Di entropy",
        "twelve_state_entropy": "12-state entropy", "mi_3di_12st": "3Di–12st MI"}
FRAME = {"s1": "same strand +1", "s2": "same strand +2", "a0": "opposite a0", "a1": "opposite a1",
         "a2": "opposite a2"}
I = pd.read_csv(DATA / "figL05_six_frame_inheritance.tsv.gz", sep="\t")
D = pd.read_csv(DATA / "figL05_six_frame_discrimination.tsv.gz", sep="\t")
H = pd.read_csv(DATA / "figL05_six_frame_delta.tsv.gz", sep="\t")

fig = plt.figure(figsize=(19, 10))
gs = fig.add_gridspec(2, 5, height_ratios=[1.15, 1], hspace=0.45, wspace=0.35)
ax = fig.add_subplot(gs[0, :2])
M = I.pivot(index="feature", columns="frame", values="spearman").loc[list(FEAT), list(FRAME)]
sns.heatmap(M.rename(index=FEAT, columns=FRAME), annot=True, fmt=".2f", vmin=-0.5, vmax=0.5,
            cmap=sns.diverging_palette(250, 25, s=75, l=55, center="light", as_cmap=True),
            linewidths=2, linecolor="white", cbar_kws={"label": "Spearman ρ"}, ax=ax)
ax.set_xlabel("alternative frame")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=30)
panel(ax, "a", x=-0.25)

ax = fig.add_subplot(gs[0, 2:])
d = D[D.scope == "genes with all 5 alternatives"].set_index("feature").loc[list(FEAT)]
x = np.arange(len(FEAT))
w = 0.26
for i, (top, lab) in enumerate(((1, "rank 1"), (2, "top 2"), (3, "top 3"))):
    b = ax.bar(x + (i - 1) * w, d[f"top{top}_rate"], width=w, color=SLOTS[i], label=lab)
    ax.bar_label(b, fmt="%.3f", fontsize=8.5, color=INK2, padding=1)
    ax.hlines(d[f"top{top}_chance"].iloc[0], x[0] - 1.5 * w, x[-1] + 1.5 * w, colors=SLOTS[i],
              linestyles=":", lw=1.4)
ax.set_xticks(x, [FEAT[f] for f in FEAT])
ax.set_ylim(0, 1.08)
ax.set_ylabel(f"fraction of genes (n = {int(d.genes.iloc[0]):,})")
ax.legend(loc="lower right", ncol=3, fontsize=9)
panel(ax, "b")

for j, (f, col) in enumerate(zip(FEAT, SLOTS)):
    ax = fig.add_subplot(gs[1, j])
    h = H[H.feature == f]
    edges = np.r_[h.bin_left.to_numpy(), h.bin_right.to_numpy()[-1]]
    ax.stairs(h["count"].to_numpy(), edges, fill=True, color=col, alpha=0.9)
    ax.axvline(0, color=INK2, lw=1, ls="--")
    ax.set_xlabel(f"Δ {FEAT[f]}" + (" (aa)" if f == "aa_len" else " (bits)"))
    ax.set_yticks([])
    ax.set_title(f"{h.fraction_positive.iloc[0]:.1%} > 0", loc="left", fontsize=10)
    panel(ax, "cdefg"[j], x=-0.05)
save(fig, "figL05_six_frame")